<a href="https://colab.research.google.com/github/niko-vision/ab-test-analysis/blob/main/ab_test_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Test Analysis

This project analyzes the results of an A/B test using SQL, Python, and Tableau.




## Project Overview

The analysis is based on event-level data from an online store and focuses on funnel conversion metrics calculated at the session level.

The workflow includes data extraction, metric preparation, statistical testing, and result visualization in Tableau.

## Objective

The goal is to compare the **control group** with the **experimental group** and check whether the differences in key conversion metrics are statistically significant.


## Metrics

The analysis is based on four metrics:

*   `add_payment_info` / `session`
*   `add_shipping_info` / `session`
*   `begin_checkout` / `session`
*   `new_accounts` / `session`

## Method

For each test:

*   calculate conversion rate for control and experiment
*   compare the difference between groups
*   check statistical significance using a two-proportion z-test



## Expected Output

The final dataset includes:

*   **conversion rate for control and experiment**
*   **metric change (%)**
*   **z-statistic**
*   **p-value**
*   **significance flag**

# Data Extraction


 Data is extracted from BigQuery using SQL and includes session information, user attributes, and A/B test groups.

In [ ]:
#import libraries
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
#install required packages
!pip install --quiet google-cloud-bigquery pandas-gbq

In [ ]:
#authenticate user
from google.colab import auth
auth.authenticate_user()

In [ ]:
#create BigQuery client
from google.cloud import bigquery
client = bigquery.Client(project="data-analytics-mate")

## SQL Query

The query joins session data with A/B test assignments and extracts all necessary fields for analysis.

In [ ]:
query = """
WITH session_info as (
SELECT
 s.date,
 s.ga_session_id,
 sp.country,
 sp.device,
 sp.continent,
 sp.channel,
 ab.test,
 ab.test_group
FROM `DA.ab_test` ab
JOIN `DA.session` s
    ON ab.ga_session_id = s.ga_session_id
JOIN `DA.session_params` sp
    ON sp.ga_session_id = ab.ga_session_id
),
session_with_orders as (
  SELECT
   si.date,
   si.country,
   si.device,
   si.continent,
   si.channel,
   si.test,
   si.test_group,
   COUNT(DISTINCT o.ga_session_id) as session_with_orders
  FROM `DA.order` o
  JOIN session_info si
      ON o.ga_session_id = si.ga_session_id
  GROUP BY
   si.date,
   si.country,
   si.device,
   si.continent,
   si.channel,
   si.test,
   si.test_group
),
events as (
  SELECT
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     ep.event_name,
     COUNT(ep.ga_session_id) as event_cnt
  FROM `DA.event_params` ep
  JOIN session_info si
      ON ep.ga_session_id = si.ga_session_id
  GROUP BY
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     ep.event_name
),
session as (
    SELECT
        si.date,
        si.country,
        si.device,
        si.continent,
        si.channel,
        si.test,
        si.test_group,
        COUNT(DISTINCT si.ga_session_id) as session_cnt
     FROM session_info si
     GROUP BY
        si.date,
        si.country,
        si.device,
        si.continent,
        si.channel,
        si.test,
        si.test_group
),
account as (
  SELECT
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group,
     COUNT(DISTINCT acs.ga_session_id) as new_account_cnt
  FROM `DA.account_session` acs
  JOIN session_info si
      ON acs.ga_session_id = si.ga_session_id
  GROUP BY
     si.date,
     si.country,
     si.device,
     si.continent,
     si.channel,
     si.test,
     si.test_group
)


SELECT
     swor.date,
     swor.country,
     swor.device,
     swor.continent,
     swor.channel,
     swor.test,
     swor.test_group,
     'session with orders' as event_name,
     swor.session_with_orders as value
FROM session_with_orders swor

UNION ALL

SELECT
     ev.date,
     ev.country,
     ev.device,
     ev.continent,
     ev.channel,
     ev.test,
     ev.test_group,
     ev.event_name,
     ev.event_cnt as value
FROM events ev

UNION ALL

SELECT
     ses.date,
     ses.country,
     ses.device,
     ses.continent,
     ses.channel,
     ses.test,
     ses.test_group,
     'session' as event_name,
     ses.session_cnt as value
FROM session ses

UNION ALL

SELECT
     acc.date,
     acc.country,
     acc.device,
     acc.continent,
     acc.channel,
     acc.test,
     acc.test_group,
     'new account' as event_name,
     acc.new_account_cnt as value
FROM account acc
"""
#execute query and load data
df = client.query(query).to_dataframe()
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-01,Uruguay,mobile,Americas,Organic Search,2,2,session with orders,1
1,2020-11-01,Malta,mobile,Europe,Social Search,2,2,session with orders,1
2,2020-11-03,Macao,mobile,Asia,Organic Search,2,2,session with orders,1
3,2020-11-03,El Salvador,desktop,Americas,Paid Search,2,2,session with orders,1
4,2020-11-04,Dominican Republic,mobile,Americas,Organic Search,2,2,session with orders,1



> *The loaded dataset was reviewed to confirm its structure and validate the event values used in the analysis.*


# Data Preparation


Only the required events are kept, and the data is aggregated by test, group, and event type.

In [ ]:
#aggregate data

df_agg = df.groupby(['test', 'test_group', 'event_name'])['value'].sum().reset_index()
df_agg.head(10)

,test,test_group,event_name,value
0,1,1,add_payment_info,1988
1,1,1,add_shipping_info,3034
2,1,1,add_to_cart,1395
3,1,1,begin_checkout,3784
4,1,1,click,368
5,1,1,first_visit,30596
6,1,1,new account,3823
7,1,1,page_view,191543
8,1,1,scroll,73244
9,1,1,select_item,543


In [ ]:
#define required events
needed_events = [
                 'session',
                 'add_payment_info',
                 'add_shipping_info',
                 'begin_checkout',
                 'new account'
                 ]

#filter required events
df_needed = df[df['event_name'].isin(needed_events)]
df_needed.head()

,date,country,device,continent,channel,test,test_group,event_name,value
117,2020-11-03,Qatar,mobile,Asia,Organic Search,2,2,new account,1
118,2020-11-03,Ecuador,mobile,Americas,Direct,2,2,new account,1
119,2020-11-12,New Zealand,mobile,Oceania,Undefined,2,2,new account,1
120,2020-11-12,Bulgaria,mobile,Europe,Paid Search,2,2,new account,1
121,2020-11-15,Bulgaria,desktop,Europe,Social Search,2,2,new account,1


In [ ]:
#check filtered events
df_needed['event_name'].unique()

array(['new account', 'session', 'begin_checkout', 'add_payment_info',
       'add_shipping_info'], dtype=object)

In [ ]:
#map metrics
metrics = {
           'add_payment_info / session': 'add_payment_info',
           'add_shipping_info / session': 'add_shipping_info',
           'begin_checkout / session': 'begin_checkout',
           'new_accounts / session': 'new account'
           }

In [ ]:
session_df = df_agg[df_agg['event_name'] == 'session'].copy()                    #filter - session,work with copy
session_df = session_df.rename(columns={'value': 'session_value'})               #rename column
session_df = session_df[['test', 'test_group', 'session_value']]                 #keep needed columns

metric_tables = []   # create empty list

for metric_name, event in metrics.items():                        #loop over metrics
    event_df = df_agg[df_agg['event_name'] == event].copy()       #filter - event
    event_df = event_df.rename(columns={'value': 'event_value'})  #rename column
    event_df = event_df[['test', 'test_group', 'event_value']]    #keep needed columns

    merged_df = event_df.merge(session_df, on=['test', 'test_group'], how='left')         #merge with sessions
    merged_df['metric'] = metric_name                                                     #add metric name
    merged_df['conversion_rate'] = merged_df['event_value'] / merged_df['session_value']  #calculate CR

    metric_tables.append(merged_df)   #save result

metrics_df = pd.concat(metric_tables, ignore_index=True)   #concatenate all
metrics_df.head(5)

,test,test_group,event_value,session_value,metric,conversion_rate
0,1,1,1988,45362,add_payment_info / session,0.043825
1,1,2,2229,45193,add_payment_info / session,0.049322
2,2,1,2344,50637,add_payment_info / session,0.04629
3,2,2,2409,50244,add_payment_info / session,0.047946
4,3,1,3623,70047,add_payment_info / session,0.051722


# Statistical Testing
  
  
Group mapping used in this analysis:
- test_group = **1** - **control**
- test_group = **2** - **experiment**


The control and experiment groups are compared to measure metric change and test statistical significance.

## Group Comparison

In [ ]:
#pivot control and experiment
comparison_df = metrics_df.pivot(
                                 index=['test', 'metric'],
                                 columns='test_group',
                                 values=['event_value', 'session_value', 'conversion_rate']
                                ).reset_index()

comparison_df.head()

test                       metric event_value       session_value  \
test_group                                             1     2             1   
0             1   add_payment_info / session        1988  2229         45362   
1             1  add_shipping_info / session        3034  3221         45362   
2             1     begin_checkout / session        3784  4021         45362   
3             1       new_accounts / session        3823  3681         45362   
4             2   add_payment_info / session        2344  2409         50637   

                  conversion_rate            
test_group      2               1         2  
0           45193        0.043825  0.049322  
1           45193        0.066884  0.071272  
2           45193        0.083418  0.088974  
3           45193        0.084278  0.081451  
4           50244         0.04629  0.047946

In [ ]:
#rename output columns

comparison_df.columns = [
                        'test',
                        'metric',
                        'event_cc',          #cc = control
                        'event_ev',          #ev = experiment
                        'session_cc',
                        'session_ev',
                        'cr_cc',
                        'cr_ev'
                        ]

comparison_df.head()

,test,metric,event_cc,event_ev,session_cc,session_ev,cr_cc,cr_ev
0,1,add_payment_info / session,1988,2229,45362,45193,0.043825,0.049322
1,1,add_shipping_info / session,3034,3221,45362,45193,0.066884,0.071272
2,1,begin_checkout / session,3784,4021,45362,45193,0.083418,0.088974
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451
4,2,add_payment_info / session,2344,2409,50637,50244,0.04629,0.047946


In [ ]:
#calculate metric change

comparison_df['metric_change_pct'] = (
                                      (comparison_df['cr_ev'] - comparison_df['cr_cc']) / comparison_df['cr_cc']
                                     ) * 100

comparison_df.head()

,test,metric,event_cc,event_ev,session_cc,session_ev,cr_cc,cr_ev,metric_change_pct
0,1,add_payment_info / session,1988,2229,45362,45193,0.043825,0.049322,12.542021
1,1,add_shipping_info / session,3034,3221,45362,45193,0.066884,0.071272,6.560481
2,1,begin_checkout / session,3784,4021,45362,45193,0.083418,0.088974,6.660587
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451,-3.354299
4,2,add_payment_info / session,2344,2409,50637,50244,0.04629,0.047946,3.576911


## Z-Test Calculation

In [ ]:
#calculate z-test
z_stats = []
p_values = []

for _, row in comparison_df.iterrows():                #loop over rows
    count = [row['event_ev'], row['event_cc']]         #set event counts
    nobs = [row['session_ev'], row['session_cc']]      #set sample sizes

    z_stat, p_value = proportions_ztest(count, nobs)    #calculate z-test

    #append results
    z_stats.append(z_stat)
    p_values.append(p_value)

#save results
comparison_df['z_stat'] = z_stats
comparison_df['p_value'] = p_values
comparison_df['significant'] = comparison_df['p_value'] < 0.05  #add significance flag

comparison_df.head()


,test,metric,event_cc,event_ev,session_cc,session_ev,cr_cc,cr_ev,metric_change_pct,z_stat,p_value,significant
0,1,add_payment_info / session,1988,2229,45362,45193,0.043825,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info / session,3034,3221,45362,45193,0.066884,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout / session,3784,4021,45362,45193,0.083418,0.088974,6.660587,2.978783,0.002894,True
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info / session,2344,2409,50637,50244,0.04629,0.047946,3.576911,1.240994,0.214608,False


In [ ]:
#manual check for test 1

comparison_df[comparison_df['test'] == 1]

,test,metric,event_cc,event_ev,session_cc,session_ev,cr_cc,cr_ev,metric_change_pct,z_stat,p_value,significant
0,1,add_payment_info / session,1988,2229,45362,45193,0.043825,0.049322,12.542021,3.924884,0.000087,True
1,1,add_shipping_info / session,3034,3221,45362,45193,0.066884,0.071272,6.560481,2.603571,0.009226,True
2,1,begin_checkout / session,3784,4021,45362,45193,0.083418,0.088974,6.660587,2.978783,0.002894,True
3,1,new_accounts / session,3823,3681,45362,45193,0.084278,0.081451,-3.354299,-1.542883,0.122859,False


In [ ]:
#rename columns for final output

final_df = comparison_df.rename(columns={
                                         'test': 'test_number',
                                         'event_cc': 'control_events',
                                         'session_cc': 'control_sessions',
                                         'cr_cc': 'control_rate',
                                         'event_ev': 'experiment_events',
                                         'session_ev': 'experiment_sessions',
                                         'cr_ev': 'experiment_rate'
                                        })


In [ ]:
#round final values

final_df['control_rate'] = final_df['control_rate'].round(4)
final_df['experiment_rate'] = final_df['experiment_rate'].round(4)
final_df['metric_change_pct'] = final_df['metric_change_pct'].round(2)
final_df['z_stat'] = final_df['z_stat'].round(4)
final_df['p_value'] = final_df['p_value'].round(4)

In [ ]:
#check final columns

final_df.columns

Index(['test_number', 'metric', 'control_events', 'experiment_events',
       'control_sessions', 'experiment_sessions', 'control_rate',
       'experiment_rate', 'metric_change_pct', 'z_stat', 'p_value',
       'significant'],
      dtype='object')

# Final Results

The final output of the analysis is a structured table with conversion rates for control and experiment, metric change, z-statistics, p-values, and significance flags.

This dataset was exported as a CSV file and used for the final Tableau dashboard.

In [ ]:
#mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#save final dataset
final_df.to_csv('/content/drive/MyDrive/ab_test_results.csv', index=False)

# Project Links

- [**Tableau Dashboard**](https://public.tableau.com/views/ABTestAnalysis_17739424276880/KeyMetricsResults?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)
- [**Calculated metrics CSV**](https://drive.google.com/file/d/1RBr6neu19NwAScuH3NlxNUOZ4bskuu5D/view?usp=sharing)

## Final Conclusion

This analysis compared control and experimental variants across four A/B tests using funnel conversion metrics and statistical significance testing.

The results were not consistent across the experiments:

*   Test 1 showed statistically significant improvement in several key funnel stages
*   Test 2 showed no statistically significant differences
*   Test 3 showed a statistically significant decline in Begin Checkout
*   Test 4 showed statistically significant negative effects in both Begin Checkout and New Accounts


Overall, the findings do not provide enough evidence to conclude that the experimental variant is consistently better than the control version.

Since the results vary across tests and include statistically significant negative effects, the treatment should not be recommended for full rollout without additional validation and deeper investigation.